# Sales forecasting experiment

Compare a moving-average baseline, linear trend, and random-forest time-index model on the same chronological holdout. This is a small reproducible experiment, not a claim that one model will generalize to other datasets.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

root = Path.cwd()
if not (root / "datasets").exists(): root = root.parent
df = pd.read_csv(root / "datasets" / "autods_demo_sales.csv", parse_dates=["Order Date"])
weekly = (df.dropna(subset=["Order Date", "Revenue"])
          .set_index("Order Date")["Revenue"].resample("W").sum().astype(float))
if len(weekly) < 16:
    raise ValueError(f"Need at least 16 weekly observations; found {len(weekly)}")
test_n = max(4, min(round(len(weekly) * 0.2), len(weekly) - 12))
train, test = weekly.iloc[:-test_n], weekly.iloc[-test_n:]
print(f"{len(train)} training and {len(test)} chronological test weeks; test starts {test.index.min().date()}")
weekly.tail()

In [ ]:
def predictions(model_name, history, horizon):
    x_train = np.arange(len(history)).reshape(-1, 1)
    x_future = np.arange(len(history), len(history) + horizon).reshape(-1, 1)
    if model_name == "moving_average":
        return np.repeat(history.tail(min(4, len(history))).mean(), horizon)
    model = LinearRegression() if model_name == "linear_regression" else RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42)
    model.fit(x_train, history.to_numpy())
    return np.clip(model.predict(x_future), 0, None)

def score(actual, predicted):
    nonzero = np.abs(actual) > 1e-9
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAPE_pct": np.mean(np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero])) * 100 if nonzero.any() else np.nan,
        "R2": r2_score(actual, predicted) if len(actual) > 1 else np.nan,
    }

results = []
for name in ["moving_average", "linear_regression", "random_forest"]:
    pred = predictions(name, train, len(test))
    results.append({"model": name, **score(test.to_numpy(), pred)})
scores = pd.DataFrame(results).sort_values("MAE")
display(scores)
display(pd.DataFrame({"actual": test, "linear_regression": predictions("linear_regression", train, len(test))}, index=test.index).plot(title="Weekly revenue: chronological holdout"))